In [0]:
%run ../00_config/paths

In [0]:
%sql
DROP SCHEMA IF EXISTS adbolist26.gold

In [0]:

orders = spark.read.format("delta").load(SILVER_ORDERS_PATH)
order_items = spark.read.format("delta").load(SILVER_ORDER_ITEMS_PATH)
order_payments = spark.read.format("delta").load(SILVER_ORDER_PAYMENTS_PATH)
order_reviews = spark.read.format("delta").load(SILVER_ORDER_REVIEWS_PATH)
customers = spark.read.format("delta").load(SILVER_CUSTOMERS_PATH)
products = spark.read.format("delta").load(SILVER_PRODUCTS_PATH)
sellers = spark.read.format("delta").load(SILVER_SELLERS_PATH)
geolocation = spark.read.format("delta").load(SILVER_GEOLOCATION_PATH)
product_translation = spark.read.format("delta").load(SILVER_CATEGORY_TRANSLATION_PATH)

In [0]:
from pyspark.sql import functions as F

payments_agg = (
    order_payments
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("total_payment_value"),
        F.max("payment_installments").alias("payment_installments"),
        F.first("payment_type").alias("payment_type")
    )
)

In [0]:
reviews_clean = (
    order_reviews
    .groupBy("order_id")
    .agg(
        F.max("review_score").alias("review_score"),
        F.first("review_comment_message").alias("review_comment_message")
    )
)

In [0]:
from pyspark.sql import Row

states = [
    Row(state="SP", state_name="São Paulo"),
    Row(state="RJ", state_name="Rio de Janeiro"),
    Row(state="MG", state_name="Minas Gerais"),
    Row(state="BA", state_name="Bahia"),
    Row(state="PR", state_name="Paraná"),
    Row(state="RS", state_name="Rio Grande do Sul"),
    Row(state="SC", state_name="Santa Catarina"),
    Row(state="PE", state_name="Pernambuco"),
    Row(state="CE", state_name="Ceará"),
    Row(state="GO", state_name="Goiás"),
    Row(state="ES", state_name="Espírito Santo"),
    Row(state="MT", state_name="Mato Grosso"),
    Row(state="MS", state_name="Mato Grosso do Sul"),
    Row(state="DF", state_name="Distrito Federal"),
    Row(state="AM", state_name="Amazonas"),
    Row(state="PA", state_name="Pará"),
    Row(state="MA", state_name="Maranhão"),
    Row(state="PI", state_name="Piauí"),
    Row(state="RN", state_name="Rio Grande do Norte"),
    Row(state="AL", state_name="Alagoas"),
    Row(state="PB", state_name="Paraíba"),
    Row(state="SE", state_name="Sergipe"),
    Row(state="TO", state_name="Tocantins"),
    Row(state="RO", state_name="Rondônia"),
    Row(state="AC", state_name="Acre"),
    Row(state="AP", state_name="Amapá"),
    Row(state="RR", state_name="Roraima")
]

states_df = spark.createDataFrame(states)

In [0]:

gold_df = (
    order_items.alias("oi")
    .join(orders.alias("o"), "order_id")
    .join(customers.alias("c"), "customer_id")
    .join(products.alias("p"), "product_id", "left")
    .join(states_df.alias("st"),F.col("customer_state") == F.col("st.state"),"left")
    .join(product_translation.alias("t"),
          F.col("p.product_category_name") == F.col("t.product_category_name"),
          "left")
    .join(sellers.alias("s"), "seller_id", "left")
    .join(payments_agg.alias("pay"), "order_id", "left")
    .join(reviews_clean.alias("r"), "order_id", "left")
)

In [0]:
gold_df = gold_df.withColumn(
    "total_revenue",
    F.col("price") + F.col("freight_value")
).withColumn(
    "delivery_time_days",
    F.datediff("order_delivered_customer_date", "order_purchase_timestamp")
).withColumn(
    "approved_time_days",
    F.datediff("order_approved_at", "order_purchase_timestamp")
).withColumn(
    "real_time_days",
    F.datediff("order_delivered_customer_date", "order_purchase_timestamp")
).withColumn(
    "delay_days",
    F.datediff("order_delivered_customer_date", "order_estimated_delivery_date")
).withColumn(
    "is_delayed",
    F.when(F.col("delay_days") > 0, 1).otherwise(0)
).withColumn(
    "is_5_star",
    F.when(F.col("review_score") == 5, 1).otherwise(0)
).withColumn(
    "product_volume_cm3",
    F.col("product_length_cm") *
    F.col("product_height_cm") *
    F.col("product_width_cm")
).withColumn(
    "purchase_year",
    F.year("order_purchase_timestamp")
).withColumn(
    "purchase_month",
    F.month("order_purchase_timestamp")
).withColumn(
    "purchase_quarter",
    F.quarter("order_purchase_timestamp")
)

In [0]:
from pyspark.sql import functions as F

gold_df = (
    gold_df
    .select(
        # Identificadores
        "order_id",
        "order_item_id",
        "customer_id",
        "customer_unique_id",
        "product_id",
        "seller_id",

        # Fechas
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",

        # Métricas
        "price",
        "freight_value",
        "total_revenue",
        "total_payment_value",
        "payment_installments",
        "payment_type",

        # Review
        "review_score",
        "is_5_star",

        # Cliente
        "customer_city",
        "customer_state",

        # Seller
        "seller_city",
        "seller_state",

        #State completo
        F.col("st.state_name").alias("customer_state_full"),

        # Producto (renombramos correctamente)
        F.col("p.product_category_name").alias("product_category_name_portuguese"),
        F.col("t.product_category_name_english").alias("product_category_name_english"),
        "product_weight_g",
        "product_volume_cm3",

        # Derivadas
        "delivery_time_days",
        "delay_days",
        "is_delayed",
        "purchase_year",
        "purchase_month",
        "purchase_quarter",
        "approved_time_days",
        "real_time_days"
    )
)

In [0]:
%sql
 CREATE SCHEMA IF NOT EXISTS adbolist26.gold

In [0]:
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("path", GOLD_PATH) \
    .saveAsTable("gold.analytics")